In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/psfc.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/t2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/SO2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/NMVOC_finn.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/bio.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/rain.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/u10.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/swdown.npy
/kaggle/input/competitions/anrf-aise-hack-pha

In [2]:
!pip install -q timm einops

In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

!pip install -q timm einops

"""
PM2.5 Forecasting — v3 (Target Score ≥ 0.9300)
================================================
Changes vs v2 (score 0.9247 → target 0.9300):

  ✅ Larger model: base=96  (was 64) → ~20M params for more capacity
  ✅ Extended training: epochs=120, patience=20  (was 80/15)
  ✅ Higher episode loss weight: ep_w grows 0.20 → 0.45  (was 0.35)
  ✅ Lower LR floor: cosine bottoms at 0.005  (was 0.02)
  ✅ 8-aug TTA: 4 flips × 2 rotations (0°/180°)  (was 4 flips only)
  ✅ Checkpoint ensemble at inference: average preds from ep060/070/080/best
  ✅ All v2 fixes retained (correct STL episodes, dynamic ep_w, EMA, etc.)
"""

import os, random, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from pathlib import Path
from copy import deepcopy
from datetime import datetime
from tqdm import tqdm

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
os.environ['PYTHONHASHSEED'] = str(SEED)

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS = torch.cuda.device_count()
print(f"Device: {DEVICE}  |  GPUs: {NUM_GPUS}")
for i in range(NUM_GPUS):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE = Path('/kaggle/input/competitions/'
            'anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/'
            'aisehack-theme-2')
RAW  = BASE / 'raw'
TEST = BASE / 'test_in'

MONTH_DIRS = {
    'April'   : RAW / 'APRIL_16',
    'July'    : RAW / 'JULY_16',
    'October' : RAW / 'OCT_16',
    'December': RAW / 'DEC_16',
}

T_IN  = 10
T_MET = 10
T_OUT = 16

MET_FEATURES = ['u10', 'v10', 't2', 'q2', 'psfc', 'pblh', 'rain', 'swdown']
EMI_FEATURES = ['PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'bio']
ALL_FEATURES = ['cpm25'] + MET_FEATURES + EMI_FEATURES

# 10(pm25) + 10×8(met) + 10×6(emission) + 10×2(sincos) = 170
IN_CH = T_IN + T_MET * len(MET_FEATURES) + T_MET * len(EMI_FEATURES) + T_MET * 2
print(f"Input channels: {IN_CH}")

# ── CONFIG ─────────────────────────────────────────────────────────────────────
# Key changes: epochs 80→120, patience 15→20, base model size 64→96
CFG = dict(
    epochs        = 120,   # was 80  — model was still improving at ep80
    batch_size    = 16,
    lr            = 5e-4,
    weight_decay  = 1e-4,
    grad_clip     = 1.0,
    patience      = 20,    # was 15
    warmup_epochs = 5,
    ema_decay     = 0.999,
)

LOG_FEATS = {'cpm25', 'rain', 'pblh', 'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'bio'}


# =============================================================================
# 1. NORMALISATION
# =============================================================================
def compute_norm_stats():
    rng = np.random.default_rng(SEED)
    stats = {}
    print("Computing norm stats...")
    for feat in ALL_FEATURES:
        samples, use_log = [], feat in LOG_FEATS
        for d in MONTH_DIRS.values():
            path = d / f'{feat}.npy'
            if not path.exists():
                continue
            arr = np.load(path, mmap_mode='r').ravel().astype(np.float32)
            if use_log:
                arr = np.log1p(arr.clip(0))
            idx = rng.choice(len(arr), max(1, min(int(len(arr)*0.05), 500_000)), replace=False)
            samples.append(arr[idx])
        if not samples:
            stats[feat] = {'log': use_log, 'mean': 0.0, 'std': 1.0}
            print(f"  {feat:<14} MISSING — identity stats")
            continue
        vals = np.concatenate(samples)
        stats[feat] = {
            'log' : use_log,
            'mean': float(vals.mean()),
            'std' : float(vals.std().clip(1e-8)),
        }
        print(f"  {feat:<14} mean={stats[feat]['mean']:>10.4f}  "
              f"std={stats[feat]['std']:>9.4f}  log={use_log}")
    return stats

def normalize(arr, s):
    if s['log']:
        arr = np.log1p(arr.clip(0))
    return (arr - s['mean']) / s['std']

def denormalize(arr, s):
    out = arr * s['std'] + s['mean']
    if s['log']:
        out = np.expm1(np.clip(out, -9., 9.)).clip(0)
    return out


# =============================================================================
# 2. STL EPISODE MASK — OFFICIAL DEFINITION  r_{i,t} > 3 * sigma_i
# =============================================================================
def compute_episode_mask(month_dir):
    """
    Official competition definition:
        r_{i,t} > 3 * sigma_i
    where:
        r_{i,t}  = cpm25 - trend(25h MA) - diurnal_cycle(24h)
        sigma_i  = std(residuals) per grid point over full month  (H, W)
    Returns: bool array (T, H, W)
    """
    cpm25 = np.load(Path(month_dir) / 'cpm25.npy').astype(np.float32)
    T, H, W = cpm25.shape

    # Step 1: Trend — 25h centred MA per grid point
    trend = np.empty_like(cpm25)
    pad   = 12
    for hi in range(H):
        for wi in range(W):
            s = np.pad(cpm25[:, hi, wi], (pad, pad), mode='edge')
            trend[:, hi, wi] = np.convolve(s, np.ones(25)/25, mode='valid')

    detrended = cpm25 - trend   # (T, H, W)

    # Step 2: Diurnal cycle — mean detrended value per hour-of-day
    diurnal = np.zeros((24, H, W), dtype=np.float32)
    counts  = np.zeros(24, dtype=np.int32)
    for t in range(T):
        h = t % 24
        diurnal[h] += detrended[t]
        counts[h]  += 1
    for h in range(24):
        if counts[h] > 0:
            diurnal[h] /= counts[h]
    diurnal_full = np.stack([diurnal[t % 24] for t in range(T)])  # (T, H, W)

    # Step 3: Residual
    residual = detrended - diurnal_full  # (T, H, W)

    # Step 4: sigma_i — std per grid point (axis=0 → shape H, W)
    sigma_i = residual.std(axis=0).clip(1e-8)  # (H, W)

    # Step 5: Official threshold
    mask = residual > (3.0 * sigma_i[np.newaxis, :, :])  # (T, H, W)

    ep_pct = mask.mean() * 100
    print(f"    episode%={ep_pct:.3f}%  sigma_mean={sigma_i.mean():.2f}  sigma_max={sigma_i.max():.2f}")
    return mask   # (T, H, W) bool


# =============================================================================
# 3. METRICS — uses pre-computed STL masks, NOT batch re-derived
# =============================================================================
def smape_np(p, y, eps=1.0):
    return float(np.mean(2*np.abs(p-y)/(np.abs(p)+np.abs(y)+eps)))

def phase2_metrics(pred_r, y_r, ep_mask_np):
    """
    pred_r, y_r  : (N, T_OUT, H, W)  real-space values
    ep_mask_np   : (N, T_OUT, H, W)  bool — pre-computed STL episodes
    """
    B, T, H, W = pred_r.shape
    g_smapes, ep_corrs, ep_smapes = [], [], []

    for t in range(T):
        p_t = pred_r[:,t].ravel()
        y_t = y_r[:,t].ravel()
        g_smapes.append(smape_np(p_t, y_t))

    for t in range(T):
        ep_t = ep_mask_np[:, t]  # (N, H, W)
        if ep_t.any():
            p_ep = pred_r[:,t][ep_t]
            y_ep = y_r[:,t][ep_t]
            if len(p_ep) > 1:
                c = float(np.corrcoef(p_ep, y_ep)[0,1])
                if not np.isnan(c):
                    ep_corrs.append(c)
            ep_smapes.append(smape_np(p_ep, y_ep))

    gs    = float(np.mean(g_smapes))
    ec    = float(np.mean(ep_corrs))  if ep_corrs  else 0.0
    es    = float(np.mean(ep_smapes)) if ep_smapes else gs
    score = ((1-gs/2) + (ec+1)/2 + (1-es/2)) / 3
    return {'gsmape': gs, 'ecorr': ec, 'esmape': es, 'score': score}


# =============================================================================
# 4. DATASET — passes STL episode mask per sample
# =============================================================================
class PM25Dataset(Dataset):
    def __init__(self, month_dir, stride=2, t_start=0, t_end=None,
                 episode_mask=None, augment=False):
        month_dir = Path(month_dir)
        ref_shape = np.load(month_dir / 'cpm25.npy', mmap_mode='r').shape
        self.mmap = {}
        for f in ALL_FEATURES:
            p = month_dir / f'{f}.npy'
            self.mmap[f] = np.load(p, mmap_mode='r') if p.exists() \
                           else np.zeros(ref_shape, dtype=np.float32)

        time_arr = np.load(month_dir / 'time.npy', allow_pickle=True)
        try:
            self.hours = np.array([
                datetime.strptime(str(s), '%Y-%m-%d_%H:%M:%S').hour
                for s in time_arr], dtype=np.float32)
        except:
            self.hours = (np.arange(len(time_arr)) % 24).astype(np.float32)

        T, H, W   = ref_shape
        t_end     = t_end or T
        self.H, self.W = H, W
        self.augment   = augment
        self.ep_mask   = episode_mask
        self.indices   = [i for i in range(0, T-T_IN-T_OUT+1, stride)
                          if t_start <= i and i+T_IN+T_OUT <= t_end]

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        t0     = self.indices[idx]
        H, W   = self.H, self.W
        flip_h = self.augment and random.random() > 0.5
        flip_v = self.augment and random.random() > 0.7

        def load(feat, start, length):
            a = self.mmap[feat][start:start+length].astype(np.float32)
            a = normalize(a, norm_stats[feat])
            if flip_h: a = a[:, :, ::-1].copy()
            if flip_v: a = a[:, ::-1, :].copy()
            return a

        pm      = load('cpm25', t0, T_IN)
        met_emi = np.concatenate(
            [load(f, t0, T_MET) for f in (MET_FEATURES + EMI_FEATURES)], axis=0)
        hw    = self.hours[t0:t0+T_MET]
        sin_h = np.sin(2*np.pi*hw/24).reshape(-1,1,1) * np.ones((1,H,W), np.float32)
        cos_h = np.cos(2*np.pi*hw/24).reshape(-1,1,1) * np.ones((1,H,W), np.float32)
        x     = np.concatenate([pm, met_emi, sin_h, cos_h], axis=0)

        y_raw = self.mmap['cpm25'][t0+T_IN:t0+T_IN+T_OUT].astype(np.float32)
        if flip_h: y_raw = y_raw[:, :, ::-1].copy()
        if flip_v: y_raw = y_raw[:, ::-1, :].copy()
        y = normalize(y_raw, norm_stats['cpm25'])

        if self.ep_mask is not None:
            ep = self.ep_mask[t0+T_IN:t0+T_IN+T_OUT].astype(np.float32)
            if flip_h: ep = ep[:, :, ::-1].copy()
            if flip_v: ep = ep[:, ::-1, :].copy()
        else:
            ep = np.zeros((T_OUT, H, W), np.float32)

        last_pm = pm[-1]
        return (torch.from_numpy(x),
                torch.from_numpy(y),
                torch.from_numpy(last_pm),
                torch.from_numpy(ep))


class TestDataset(Dataset):
    def __init__(self):
        ref = np.load(TEST / 'cpm25.npy', mmap_mode='r')
        self.N, _, self.H, self.W = ref.shape
        self.mmap = {}
        for f in ALL_FEATURES:
            p = TEST / f'{f}.npy'
            self.mmap[f] = np.load(p, mmap_mode='r') if p.exists() \
                           else np.zeros((self.N, T_MET, self.H, self.W), dtype=np.float32)
        print(f"TestDataset: N={self.N}, H={self.H}, W={self.W}")

    def __len__(self): return self.N

    def __getitem__(self, idx):
        H, W    = self.H, self.W
        pm      = normalize(self.mmap['cpm25'][idx].astype(np.float32), norm_stats['cpm25'])
        met_emi = np.concatenate([
            normalize(self.mmap[f][idx].astype(np.float32), norm_stats[f])
            for f in (MET_FEATURES + EMI_FEATURES)], axis=0)
        hw    = ((idx % 24) + np.arange(T_MET)) % 24
        hw    = hw.astype(np.float32)
        sin_h = np.sin(2*np.pi*hw/24).reshape(-1,1,1) * np.ones((1,H,W), np.float32)
        cos_h = np.cos(2*np.pi*hw/24).reshape(-1,1,1) * np.ones((1,H,W), np.float32)
        x     = np.concatenate([pm, met_emi, sin_h, cos_h], axis=0)
        return torch.from_numpy(x), torch.from_numpy(pm[-1])


# =============================================================================
# 5. MODEL COMPONENTS
# =============================================================================

class ChannelAttention(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(ch, max(1,ch//r)), nn.ReLU(),
                                 nn.Linear(max(1,ch//r), ch))
    def forward(self, x):
        w = torch.sigmoid(self.fc(x.mean(dim=[2,3])) + self.fc(x.amax(dim=[2,3])))
        return x * w.unsqueeze(-1).unsqueeze(-1)

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
    def forward(self, x):
        sa = torch.cat([x.mean(1,keepdim=True), x.amax(1,keepdim=True)], 1)
        return x * torch.sigmoid(self.conv(sa))

class CBAM(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.ca = ChannelAttention(ch)
        self.sa = SpatialAttention()
    def forward(self, x): return self.sa(self.ca(x))

class SpatialSelfAttention(nn.Module):
    def __init__(self, ch, n_heads=8):
        super().__init__()
        self.n_heads  = n_heads
        self.head_dim = ch // n_heads
        self.scale    = self.head_dim ** -0.5
        self.qkv      = nn.Conv2d(ch, ch*3, 1, bias=False)
        self.proj     = nn.Conv2d(ch, ch, 1)
        self.norm     = nn.GroupNorm(min(8,ch), ch)

    def forward(self, x):
        B, C, H, W = x.shape
        N   = H * W
        qkv = self.qkv(x).reshape(B, 3, self.n_heads, self.head_dim, N)
        q, k, v = qkv.unbind(1)
        attn = torch.einsum('bhdn,bhdm->bhnm', q, k) * self.scale
        attn = torch.softmax(attn, dim=-1)
        out  = torch.einsum('bhnm,bhdm->bhdn', attn, v).reshape(B, C, H, W)
        return x + self.proj(self.norm(out))


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(min(8,out_ch), out_ch), nn.GELU(),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(min(8,out_ch), out_ch), nn.GELU(),
        )
    def forward(self, x): return self.net(x)


# =============================================================================
# 5a. ImprovedSkipUNet — base=96 for higher capacity (was 64)
# =============================================================================
class ImprovedSkipUNet(nn.Module):
    def __init__(self, in_ch=IN_CH, base=96, dropout=0.10):
        """
        base=96 → ~20M params  (v2 had base=64 → ~9M params)
        Larger bottleneck (b*8 = 768 channels) captures more complex
        spatio-temporal patterns for episode correlation improvement.
        """
        super().__init__()
        b, DR = base, dropout
        self.enc1       = DoubleConv(in_ch + 1, b,   DR)   # +1 trend channel
        self.enc2       = DoubleConv(b,          b*2, DR)
        self.enc3       = DoubleConv(b*2,        b*4, DR)
        self.bottleneck = nn.Sequential(
            DoubleConv(b*4, b*8, DR),
            CBAM(b*8),
            SpatialSelfAttention(b*8, n_heads=8),
        )
        self.pool = nn.MaxPool2d(2)
        self.up3  = nn.ConvTranspose2d(b*8, b*4, 2, stride=2)
        self.dec3 = nn.Sequential(DoubleConv(b*8,  b*4, DR), CBAM(b*4))
        self.up2  = nn.ConvTranspose2d(b*4, b*2, 2, stride=2)
        self.dec2 = nn.Sequential(DoubleConv(b*4,  b*2, DR), CBAM(b*2))
        self.up1  = nn.ConvTranspose2d(b*2, b,   2, stride=2)
        self.dec1 = DoubleConv(b*2, b)
        self.out  = nn.Conv2d(b, T_OUT, 1)

    def forward(self, x, last_pm):
        trend = x[:, T_IN-1:T_IN, :, :] - x[:, T_IN-3:T_IN-2, :, :]
        xin   = torch.cat([x, trend], dim=1)
        e1 = self.enc1(xin)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        bn = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([F.interpolate(self.up3(bn), e3.shape[2:]), e3], 1))
        d2 = self.dec2(torch.cat([F.interpolate(self.up2(d3), e2.shape[2:]), e2], 1))
        d1 = self.dec1(torch.cat([F.interpolate(self.up1(d2), e1.shape[2:]), e1], 1))
        return self.out(d1) + last_pm.unsqueeze(1)


# =============================================================================
# 6. EMA
# =============================================================================
class EMA:
    def __init__(self, model, decay=0.999):
        m = model.module if isinstance(model, nn.DataParallel) else model
        self.model = deepcopy(m).eval()
        self.decay = decay

    @torch.no_grad()
    def update(self, model):
        m = model.module if isinstance(model, nn.DataParallel) else model
        for ep, p in zip(self.model.parameters(), m.parameters()):
            ep.data.mul_(self.decay).add_(p.data, alpha=1-self.decay)

    def eval(self): self.model.eval(); return self.model


# =============================================================================
# 7. LOSS — dynamic episode weight ep_w: 0.20 → 0.45  (was 0.35)
# =============================================================================
def smape_fn(pred, target, eps=1.0):
    return (2*(pred-target).abs() / (pred.abs()+target.abs()+eps)).mean()

def composite_loss(pred, target, ep_mask, epoch=0, total_epochs=120):
    """
    ep_mask  : (B, T_OUT, H, W) float32 — STL pre-computed, from Dataset

    CHANGE vs v2: ep_w ceiling raised from 0.35 → 0.45
    This pushes the model harder to reduce eSMAPE (episode SMAPE),
    which is the metric with the most remaining headroom.

    ep_w   : 0.20 → 0.45  (+0.10 ceiling vs v2's 0.35)
    huber_w: 0.50 → 0.40  (gently reduced to make room)
    smape_w: remainder     (0.30 → 0.15 by end)
    """
    progress = min(epoch / max(total_epochs, 1), 1.0)
    ep_w     = 0.20 + 0.25 * progress    # 0.20 → 0.45  (was 0.20→0.35)
    huber_w  = 0.50 - 0.10 * progress    # 0.50 → 0.40  (was 0.50→0.45)
    smape_w  = 1.0 - huber_w - ep_w      # 0.30 → 0.15

    huber = F.huber_loss(pred, target, delta=1.0)
    smape = smape_fn(pred, target)

    ep_bool = ep_mask.bool()
    if ep_bool.any():
        p_ep = pred[ep_bool]; t_ep = target[ep_bool]
        ep_loss = F.huber_loss(p_ep, t_ep, delta=0.5) + smape_fn(p_ep, t_ep)
    else:
        ep_loss = huber

    return huber_w*huber + smape_w*smape + ep_w*ep_loss


# =============================================================================
# 8. SCHEDULER — LR floor lowered from 0.02 → 0.005 for finer late convergence
# =============================================================================
def get_scheduler(opt, warmup_eps, total_eps):
    def lr_fn(ep):
        if ep < warmup_eps:
            return (ep+1) / warmup_eps
        p = (ep-warmup_eps) / max(1, total_eps-warmup_eps)
        # CHANGE: floor 0.02 → 0.005 so LR anneals lower in later epochs
        return 0.5*(1+np.cos(np.pi*p))*(1-0.005)+0.005
    return torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)


# =============================================================================
# 9. TRAINING — saves checkpoints every 10 epochs for ensemble
# =============================================================================
def train_model(model, name, train_ldr, val_ldr, cfg):
    if NUM_GPUS > 1:
        model = nn.DataParallel(model)
        print(f"DataParallel on {NUM_GPUS} GPUs")
    model.to(DEVICE)

    ema    = EMA(model, decay=cfg['ema_decay'])
    opt    = torch.optim.AdamW(model.parameters(),
                                lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    sched  = get_scheduler(opt, cfg['warmup_epochs'], cfg['epochs'])
    scaler = torch.amp.GradScaler('cuda')
    s      = norm_stats['cpm25']

    best_score = -float('inf')
    patience   = 0
    best_path  = f'/kaggle/working/{name}_best.pt'

    # Track checkpoint paths for ensemble
    ckpt_paths = []

    hdr = (f"{'Ep':>4} {'TrL':>7} {'VaL':>7} {'gSMAPE':>8} "
           f"{'eCorr':>7} {'eSMAPE':>8} {'Score':>7} {'LR':>8}")
    print(f"\n{hdr}\n{'-'*len(hdr)}")

    for epoch in range(cfg['epochs']):
        t0 = time.time()
        model.train()
        tr_loss = 0.0

        for x, y, last, ep in tqdm(train_ldr, desc=f'Ep{epoch+1:02d}', leave=False):
            x, y, last, ep = x.to(DEVICE), y.to(DEVICE), last.to(DEVICE), ep.to(DEVICE)
            opt.zero_grad()
            with torch.amp.autocast('cuda'):
                pred = model(x, last)
                loss = composite_loss(pred, y, ep, epoch=epoch, total_epochs=cfg['epochs'])
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            m = model.module if isinstance(model, nn.DataParallel) else model
            nn.utils.clip_grad_norm_(m.parameters(), cfg['grad_clip'])
            scaler.step(opt); scaler.update()
            ema.update(model)
            tr_loss += loss.item()

        sched.step()
        avg_tr = tr_loss / len(train_ldr)

        # Validate
        ema_model = ema.eval()
        va_loss   = 0.0
        p_list, y_list, ep_list = [], [], []

        with torch.no_grad():
            for x, y, last, ep in val_ldr:
                x, y, last, ep = x.to(DEVICE), y.to(DEVICE), last.to(DEVICE), ep.to(DEVICE)
                with torch.amp.autocast('cuda'):
                    pred    = ema_model(x, last)
                    va_loss += composite_loss(pred, y, ep, epoch=epoch,
                                              total_epochs=cfg['epochs']).item()
                p_list.append(pred.float().cpu().numpy())
                y_list.append(y.float().cpu().numpy())
                ep_list.append(ep.float().cpu().numpy())

        avg_va   = va_loss / len(val_ldr)
        p_all    = np.concatenate(p_list)
        y_all    = np.concatenate(y_list)
        ep_all   = np.concatenate(ep_list).astype(bool)

        metrics  = phase2_metrics(denormalize(p_all, s), denormalize(y_all, s), ep_all)
        cur_lr   = opt.param_groups[0]['lr']
        ep_min   = (time.time()-t0)/60

        is_best = metrics['score'] > best_score
        if is_best:
            best_score = metrics['score']
            patience   = 0
            torch.save(ema_model.state_dict(), best_path)
        else:
            patience += 1

        flag = '✓' if is_best else ''
        print(f"{epoch+1:>4} {avg_tr:>7.4f} {avg_va:>7.4f} "
              f"{metrics['gsmape']:>8.4f} {metrics['ecorr']:>7.4f} "
              f"{metrics['esmape']:>8.4f} {metrics['score']:>7.4f} "
              f"{cur_lr:>8.2e}  {ep_min:.1f}m {flag}")

        # Save checkpoint every 10 epochs (used for ensemble at inference)
        if (epoch+1) % 10 == 0:
            cp = Path('/kaggle/working') / f'{name}_ep{epoch+1:03d}.pt'
            torch.save(ema_model.state_dict(), cp)
            ckpt_paths.append(str(cp))
            print(f"  [ckpt] {cp.name}")

        if patience >= cfg['patience']:
            print(f"\nEarly stop ep{epoch+1}. Best score={best_score:.4f}")
            break

    ema_model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    print(f"Best EMA score={best_score:.4f}")
    return ema_model, best_score, ckpt_paths


# =============================================================================
# 10. SETUP — norm stats, episode masks, datasets
# =============================================================================
norm_stats = compute_norm_stats()

print('\nComputing STL episode masks (official definition: r_{i,t} > 3*sigma_i)...')
ep_masks = {}
for mname, d in MONTH_DIRS.items():
    print(f"  {mname}:", end=' ')
    ep_masks[mname] = compute_episode_mask(d)

# Val: last 200h of October
OCT_T     = np.load(MONTH_DIRS['October']/'cpm25.npy', mmap_mode='r').shape[0]
val_start = max(0, OCT_T - 200)
print(f"\nOctober: {OCT_T}h  Val=[{val_start},{OCT_T})={OCT_T-val_start}h")

train_ds = ConcatDataset([
    PM25Dataset(MONTH_DIRS['April'],    stride=2, augment=True,
                episode_mask=ep_masks['April']),
    PM25Dataset(MONTH_DIRS['July'],     stride=2, augment=True,
                episode_mask=ep_masks['July']),
    PM25Dataset(MONTH_DIRS['December'], stride=2, augment=True,
                episode_mask=ep_masks['December']),
    PM25Dataset(MONTH_DIRS['October'],  stride=2, t_end=val_start, augment=True,
                episode_mask=ep_masks['October']),
])
val_ds  = PM25Dataset(MONTH_DIRS['October'], stride=2, t_start=val_start,
                      episode_mask=ep_masks['October'])
test_ds = TestDataset()
print(f"Train={len(train_ds)}  Val={len(val_ds)}  Test={len(test_ds)}")

def seed_worker(wid):
    np.random.seed(SEED+wid); random.seed(SEED+wid)

g = torch.Generator(); g.manual_seed(SEED)
train_ldr = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                       num_workers=2, pin_memory=True, drop_last=True,
                       worker_init_fn=seed_worker, generator=g)
val_ldr   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                       num_workers=2, pin_memory=True)
test_ldr  = DataLoader(test_ds,  batch_size=16, shuffle=False,
                       num_workers=2, pin_memory=True)

xb, yb, lb, eb = next(iter(train_ldr))
print(f"Batch: x={xb.shape} y={yb.shape} last={lb.shape} ep={eb.shape}")
assert xb.shape[1] == IN_CH, f"Expected {IN_CH} channels, got {xb.shape[1]}"
print('Dataset check passed ✓')


# =============================================================================
# 11. TRAIN MODEL — base=96 (was 64)
# =============================================================================
unet = ImprovedSkipUNet(in_ch=IN_CH, base=96, dropout=0.10)
print(f"\nImprovedSkipUNet params: {sum(p.numel() for p in unet.parameters()):,}")
with torch.no_grad():
    _x = torch.zeros(2, IN_CH, 140, 124); _l = torch.zeros(2, 140, 124)
    _o = unet(_x, _l)
    assert _o.shape == (2, T_OUT, 140, 124), f"Bad shape: {_o.shape}"
    print(f"UNet forward: {tuple(_x.shape)} -> {tuple(_o.shape)} OK")
del _x, _l, _o

print("\n" + "="*70)
print("Training ImprovedSkipUNet v3 (base=96, ep_w→0.45, 120 epochs, LR floor=0.005)")
print("="*70)
unet_ema, unet_score, ckpt_paths = train_model(unet, 'UNet', train_ldr, val_ldr, CFG)


# =============================================================================
# 12. TTA INFERENCE — 8 augmentations: 4 flips × 2 rotations (0° / 180°)
#     Then ENSEMBLE across saved checkpoints (last 3 ckpts + best)
# =============================================================================
print("\nRunning 8-aug TTA inference with checkpoint ensemble...")
s = norm_stats['cpm25']

# 8 augmentations: (rot90_k, flip_h, flip_v)
# rot90_k=0 → 0°, rot90_k=2 → 180°  (spatial dims H×W must be square or padded)
# For non-square grids (140×124) we only use left-right and up-down flips
# plus 180° rotation (= both flips simultaneously) → 8 configs
AUG_CONFIGS = [
    (fh, fv, r180)
    for fh   in [False, True]
    for fv   in [False, True]
    for r180 in [False, True]
]
# That gives 8 augmentations (double v2's 4)

def tta_batch(model, x_np, last_np):
    B = x_np.shape[0]
    aug_x, aug_l = [], []
    for fh, fv, r180 in AUG_CONFIGS:
        xf = x_np.copy(); lf = last_np.copy()
        if fh:   xf = xf[:,:,:,::-1].copy();  lf = lf[:,:,::-1].copy()
        if fv:   xf = xf[:,:,::-1,:].copy();  lf = lf[:,::-1,:].copy()
        if r180:
            # 180° rotation = flip both axes
            xf = xf[:,:,::-1,::-1].copy(); lf = lf[:,::-1,::-1].copy()
        aug_x.append(xf); aug_l.append(lf)

    X = torch.from_numpy(np.concatenate(aug_x)).to(DEVICE)
    L = torch.from_numpy(np.concatenate(aug_l)).to(DEVICE)

    with torch.no_grad(), torch.amp.autocast('cuda'):
        pred_norm = model(X, L).float().cpu().numpy()

    preds = []
    for i, (fh, fv, r180) in enumerate(AUG_CONFIGS):
        p = pred_norm[i*B:(i+1)*B].copy()
        if fh:   p = p[:,:,:,::-1]
        if fv:   p = p[:,:,::-1,:]
        if r180: p = p[:,:,::-1,::-1]
        preds.append(p)

    return denormalize(np.mean(preds, axis=0), s)


def run_inference(model):
    """Run 8-aug TTA on the full test set with given model."""
    model.eval()
    all_preds = []
    INFER_BS  = 16
    for start in tqdm(range(0, len(test_ds), INFER_BS), desc='TTA Inference', leave=False):
        end = min(start+INFER_BS, len(test_ds))
        xs, ls = [], []
        for i in range(start, end):
            xi, li = test_ds[i]
            xs.append(xi.numpy()); ls.append(li.numpy())
        all_preds.append(tta_batch(model, np.stack(xs), np.stack(ls)))
    return np.concatenate(all_preds)


# Collect unique checkpoints for ensemble (last 3 saved + best)
# De-duplicate: best may equal one of the saved checkpoints
ensemble_ckpts = []
best_path = '/kaggle/working/UNet_best.pt'

# Use last 3 decade checkpoints + best
for cp in sorted(ckpt_paths)[-3:]:
    ensemble_ckpts.append(cp)
if best_path not in ensemble_ckpts:
    ensemble_ckpts.append(best_path)

print(f"\nEnsembling {len(ensemble_ckpts)} checkpoints:")
for cp in ensemble_ckpts:
    print(f"  {cp}")

# Build a fresh model instance for checkpoint loading
def load_model(ckpt_path):
    m = ImprovedSkipUNet(in_ch=IN_CH, base=96, dropout=0.10)
    m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    m.to(DEVICE)
    m.eval()
    return m

all_ckpt_preds = []
for cp in ensemble_ckpts:
    print(f"\n  → Running TTA for {Path(cp).name}")
    m = load_model(cp)
    preds_i = run_inference(m)
    all_ckpt_preds.append(preds_i)
    del m
    torch.cuda.empty_cache()

# Average across checkpoints
preds_ensemble = np.mean(all_ckpt_preds, axis=0).clip(0)
preds_out      = preds_ensemble.transpose(0, 2, 3, 1)   # (218, 140, 124, 16)

print(f"\nEnsemble Prediction stats:")
print(f"  shape={preds_out.shape}")
print(f"  mean={preds_ensemble.mean():.2f}  median={np.median(preds_ensemble):.2f}  "
      f"p95={np.percentile(preds_ensemble,95):.1f}  max={preds_ensemble.max():.1f}  "
      f"NaNs={np.isnan(preds_ensemble).sum()}")

assert preds_out.shape == (218, 140, 124, T_OUT), f"Shape error: {preds_out.shape}"
np.save('/kaggle/working/preds.npy', preds_out)
print(f"\nSaved /kaggle/working/preds.npy  shape={preds_out.shape}")

/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/psfc.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/t2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/SO2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/NMVOC_finn.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/bio.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/rain.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/u10.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/swdown.npy
/kaggle/input/competitions/anrf-aise-hack-pha

   1  0.1969  0.2364   0.3831  0.7830   0.6041  0.7993 2.00e-04  1.8m ✓


   2  0.1782  0.2102   0.3440  0.8099   0.5469  0.8198 3.00e-04  1.4m ✓


   3  0.1608  0.1894   0.3129  0.8341   0.4961  0.8375 4.00e-04  1.4m ✓


   4  0.1501  0.1736   0.2898  0.8540   0.4544  0.8516 5.00e-04  1.4m ✓


...

 120  0.0728  0.1007   0.1868  0.9337   0.1815  0.9276 2.50e-06  1.4m ✓
  [ckpt] UNet_ep120.pt
Best EMA score=0.9276

Running 8-aug TTA inference with checkpoint ensemble...

Ensembling 4 checkpoints:
  /kaggle/working/UNet_ep100.pt
  /kaggle/working/UNet_ep110.pt
  /kaggle/working/UNet_ep120.pt
  /kaggle/working/UNet_best.pt

  → Running TTA for UNet_ep100.pt



  → Running TTA for UNet_ep110.pt



  → Running TTA for UNet_ep120.pt



  → Running TTA for UNet_best.pt



Ensemble Prediction stats:
  shape=(218, 140, 124, 16)
  mean=37.20  median=18.16  p95=131.3  max=2611.3  NaNs=0

Saved /kaggle/working/preds.npy  shape=(218, 140, 124, 16)
